<a href="https://colab.research.google.com/github/st4rfruit/detect/blob/main/YOLO26_modeltraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install ultralytics supervision roboflow -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 47.0/235.7 GB disk)


In [ ]:
import os
HOME = os.getcwd()
print(HOME)

/content


In [ ]:
# Allow access to personal google drive and add new folders

# Connect Google Drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True) # This will prompt for authorization.

# This will create the uavs files if they don't exist.
folders =  ["yolo-medium/"]
for folder in folders:
  path = "/content/drive/MyDrive/" + folder
  if not os.path.exists(path): # Create the folder if it does not exist
    os.mkdir(path)

Mounted at /content/drive


In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets
from google.colab import userdata

!mkdir -p /content/datasets/savedir/
!cp -r "/content/drive/MyDrive/yolo-medium/images.tar.gz" "/content/datasets/savedir/"

!cp -r "/content/drive/MyDrive/yolo-medium/labels.tar.gz" "/content/datasets/savedir/"

!tar xf /content/datasets/savedir/images.tar.gz --directory /content/datasets/savedir/

!tar xf /content/datasets/savedir/labels.tar.gz --directory /content/datasets/savedir/

/content/datasets


In [ ]:
## make the directories that yolo26 expects
!mkdir /content/datasets/train/
!mkdir /content/datasets/train/images/
!mkdir /content/datasets/train/labels/
!mkdir /content/datasets/test/
!mkdir /content/datasets/test/images/
!mkdir /content/datasets/test/labels/
!mkdir /content/datasets/val/
!mkdir /content/datasets/val/images/
!mkdir /content/datasets/val/labels/

#get the data.yaml file
!cp "/content/drive/MyDrive/yolo-medium/data.yaml" "/content/datasets/data.yaml"
!ls /content/datasets/

#move the data to the expected directories
!cp -r "/content/datasets/savedir/images/train/" "/content/datasets/train/images/"
!cp -r "/content/datasets/savedir/labels/train/" "/content/datasets/train/labels/"

!cp -r "/content/datasets/savedir/images/test/" "/content/datasets/test/images/"
!cp -r "/content/datasets/savedir/labels/test/" "/content/datasets/test/labels/"

!cp -r "/content/datasets/savedir/images/val/" "/content/datasets/val/images/"
!cp -r "/content/datasets/savedir/labels/val/" "/content/datasets/val/labels/"

!ls /content/datasets/

data.yaml  savedir  test  train  val
data.yaml  savedir  test  train  val


In [ ]:
# train from a pre-trained yolo model
!yolo task=detect mode=train model=yolo26m.pt data=/content/datasets/data.yaml \
  batch=128 epochs=150 patience=30 imgsz=640 \
  mixup=0.3 scale=0.9 plots=True

In [ ]:
# if colab restarts/new session
!cp /content/drive/MyDrive/yolo-medium/output-150/last.pt /content/last.pt


In [ ]:
# train from a checkpoint
!yolo task=detect mode=train model=/content/last.pt data=/content/datasets/data.yaml \
 batch=128 epochs=150 patience=30 imgsz=640 \
 mixup=0.3 scale=0.9 plots=True

Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.3, mode=train, model=/content/last.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=No

In [ ]:
!cp "/content/datasets/runs/detect/train/weights/best.pt" "/content/drive/MyDrive/yolo-medium/best.pt"
!cp "/content/datasets/runs/detect/train/weights/last.pt" "/content/drive/MyDrive/yolo-medium/last.pt"
!cp -r "/content/datasets/runs/detect/train/" "/content/drive/MyDrive/yolo-medium/train/"

In [ ]:
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolo-medium/best.pt")
model.export(format="onnx")


Ultralytics 8.4.104 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 67.8 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/yolo-medium/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (42.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 219ms
Prepared 4 packages in 1.61s
Installed 4 packages in 255ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.27.0
 + onnxslim==0.1.94

requirements: AutoUpdate success ✅ 2.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning: Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.
  return opset9.index(g, self, index)


ONNX: slimming with onnxslim 0.1.94...
ONNX: export success ✅ 6.3s, saved as '/content/drive/MyDrive/yolo-medium/best.onnx' (77.9 MB)

Export complete (7.3s)
Results saved to /content/drive/MyDrive/yolo-medium/best.onnx
Predict:         yolo predict task=detect model=/content/drive/MyDrive/yolo-medium/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/content/drive/MyDrive/yolo-medium/best.onnx imgsz=640 data=/content/datasets/data.yaml  
Visualize:       https://netron.app


'/content/drive/MyDrive/yolo-medium/best.onnx'